In [1]:
import numpy as np

### Topic 30: Multi-variable limits and continuity surfaces


* **Mathematical Requirement:** Gradients (derivatives) only exist if limits exist. Loss functions must project a continuous surface.
* **The NaN Threat:** Division by zero or $0/0$ limits during backpropagation will instantly corrupt the entire weight matrix with `NaN` (Not a Number) values, collapsing the model.
* **Vectorized Defense:** We evaluate limits computationally using continuous approximations and NumPy masking to handle singularities without breaking vectorized execution.

In [10]:
# weight space boundaries
w1 = np.linspace(-5,5,100)
w2 = np.linspace(-5,5,100)

W1,W2 = np.meshgrid(w1,w2)

#continuous convex Loss Surface: L(w1, w2) = w1^2 + w2^2
loss_surface = W1**2 + W2**2
print(loss_surface.shape)

(100, 100)


In [24]:
w3 = np.linspace(-10,10,500)
w4 = np.linspace(-10,10,500)

W3,W4 = np.meshgrid(w3,w4)

distance_matrix = np.sqrt(W3**2 + W4**2)

# loss_surface_1 = np.sin(distance_matrix) / distance_matrix

with np.errstate(divide='ignore',invalid='ignore'):
    loss_surface_1 = np.where(
        distance_matrix == 0,
        1.0, 
        np.sin(distance_matrix)/distance_matrix
    )

print(loss_surface_1.shape)

print(f"Value at origin: {loss_surface_1[250, 250]}") 



(500, 500)
Value at origin: 0.9998661371051673


### Topic 31: Matrix Calculus Mechanics
* **Scalar-by-Vector Derivative:** The gradient of a scalar function $f(x)$ with respect to a vector $x$ is always a vector of the same shape as $x$.
* **The Linear Rule:** If $f(x) = a^T x$, then $\nabla_x f = a$. 
* **The Quadratic Rule:** If $f(x) = x^T A x$, then $\nabla_x f = (A + A^T)x$. (If $A$ is symmetric, this becomes $2Ax$).
* **Deep Learning Engine:** In a dense neural network layer $Z = XW$, the gradient of the loss $L$ with respect to the weights $W$ is strictly $\nabla_W L = X^T \cdot dZ$. This is the core equation of Backpropagation.

In [27]:
x = np.random.randn(3,1)
A = np.random.randn(3,3)
A = A @ A.T

f_x = x.T @ A @ x
print(f"scaler output f(x): {f_x.item():.4f}")

# The backward pass (Calculating the exact gradient analytically)
# Since A is symmetric, gradient is simply 2 * A * x
grad_x = 2 * A @ x
print(f"gradient shape: {grad_x.shape}")

scaler output f(x): 2.2535
gradient shape: (3, 1)


In [39]:
# 1,000 houses, 5 features each.
X = np.random.randn(1000, 5) 

# The actual house prices.
Y = np.random.randn(1000, 1) 
# Your current neural network weights initialized to zero.
W = np.zeros((5, 1))

N = X.shape[0]
# norm = np.linalg.norm((X@W)-Y)
# f_x_1 = (1/2) * np.square(norm) mathematically correct but lower is more efficient for dl
f_x_1 = np.mean(np.square(X @ W - Y)) / N
print(f_x_1)

grad_W = X.T @ ((X @ W) - Y)
print(grad_W.shape)

0.0010467729084542886
(5, 1)


### Topic 32: Numerical Partial Derivatives & Gradient Checking
* **The Concept:** $\frac{\partial L}{\partial w_i} \approx \frac{L(w_i + \epsilon) - L(w_i - \epsilon)}{2\epsilon}$ (Centered Finite Difference).
* **The ML Application:** Used strictly for "Gradient Checking." We compare numerical gradients against analytical gradients to debug backpropagation logic.
* **The Constraint:** Calculating numerical partials is $O(N)$ for $N$ weights. We only use it for debugging small prototype networks, never for actual training.

In [40]:
# A simple objective function: L(W) = W_0^2 + 3*W_1
def loss_fn(W):
    return W[0]**2 + 3*W[1]
W = np.array([2.0,4.0])
epsilon = 1e-5
W_plus = W.copy()
W_plus += epsilon

W_minus = W.copy()
W_minus -= epsilon

numerical_partial = (loss_fn(W_plus) - loss_fn(W_minus)) / (2*epsilon)

print(f"Analytical ∂L/∂W_0 at W[0]=2: {2 * W[0]}") # 2 * 2.0 = 4.0
print(f"Numerical ∂L/∂W_0: {numerical_partial:.6f}")


Analytical ∂L/∂W_0 at W[0]=2: 4.0
Numerical ∂L/∂W_0: 7.000000


In [50]:
def loss_fn(W):
    return np.sum(np.square(W),axis = 0)
W = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]]) 
epsilon = 1e-5
identity_matrix = np.identity(5)
perturbation_matrix = identity_matrix * epsilon
W_plus = W + perturbation_matrix 
W_minus = W - perturbation_matrix

W_plus_loss = loss_fn(W_plus)
W_minus_loss = loss_fn(W_minus)

difference = ((W_plus_loss - W_minus_loss)/ (2 * epsilon)).reshape(5,1)

print("Analytical Gradient:", 2 * W)
print("Numerical Gradient:", difference)


Analytical Gradient: [[ 2.]
 [ 4.]
 [ 6.]
 [ 8.]
 [10.]]
Numerical Gradient: [[ 2.]
 [ 4.]
 [ 6.]
 [ 8.]
 [10.]]
